# SPY direction baseline: repeat the previous day

Predict the next trading session's direction to be the same as the most recently completed trading session. For a forecast issued at close **t**, the prediction uses the sign of `log(close[t] / close[t-1])`; the realized outcome is the sign of `log(close[t+1] / close[t])`.

This uses the same issue and target dates as the saved signature forecast's held-out test. There is no fitting or parameter selection. **Up** means a positive close-to-close return; **down** includes zero returns, matching the existing notebooks' actual-direction convention. There is no neutral class or adjustable direction threshold in this baseline.

All calculations use local data. The saved signature predictions supply the evaluation calendar and a cross-check of realized returns; their predicted returns are not used.

## 1. Inputs and evaluation dates

The default calendar is `results/daily_refit/predictions.csv`. Keep the first prior session before filtering the evaluation period, so its return is available for the first forecast. “Previous day” means the previous trading session, including weekends and holidays.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd()
if not (PROJECT / 'src' / 'spy_adaptive_lasso.py').exists():
    PROJECT = Path('Signature_Price_Forecast').resolve()
ROOT = PROJECT.parent
SPY_FILE = ROOT / 'SP500' / 'attention_data' / 'SPY_daily.csv'
CALENDAR_FILE = PROJECT / 'results' / 'daily_refit' / 'predictions.csv'
OUTPUT = PROJECT / 'results' / 'previous_day_direction_baseline'

spy = pd.read_csv(SPY_FILE, usecols=['date', 'close'], parse_dates=['date'])
spy = spy.set_index('date').sort_index()
if not spy.index.is_unique or not np.isfinite(spy.close).all() or (spy.close <= 0).any():
    raise ValueError('SPY requires unique dates and finite positive closes.')
calendar = pd.read_csv(CALENDAR_FILE, usecols=['issue_date', 'target_date', 'actual_return'],
                       parse_dates=['issue_date', 'target_date']).sort_values('issue_date')
if calendar.empty or not calendar.issue_date.is_unique:
    raise ValueError('The saved test calendar must contain unique issue sessions.')
issue_positions = spy.index.get_indexer(calendar.issue_date)
target_positions = spy.index.get_indexer(calendar.target_date)
if (issue_positions < 1).any() or not np.array_equal(target_positions, issue_positions + 1):
    raise ValueError('Each forecast needs a prior SPY session and the next-session target.')
print(f'Evaluation: {len(calendar)} days')
print(f'Forecast issues: {calendar.issue_date.min().date()} to {calendar.issue_date.max().date()}')
print(f'Target sessions: {calendar.target_date.min().date()} to {calendar.target_date.max().date()}')

Evaluation: 310 days
Forecast issues: 2024-01-02 to 2025-03-27
Target sessions: 2024-01-03 to 2025-03-28


## 2. Predict with the last observed return

The observation used for a prediction ends at its own issue close. The target is strictly later. Assertions verify that each target is the next local SPY session and that its return agrees with the original report.

In [2]:
close = spy.close.to_numpy(float)
previous_close = close[issue_positions - 1]
issue_close = close[issue_positions]
target_close = close[target_positions]
last_day_return = np.log(issue_close / previous_close)
actual_return = np.log(target_close / issue_close)
np.testing.assert_allclose(actual_return, calendar.actual_return.to_numpy(float),
                           rtol=1e-10, atol=1e-12)

predicted_up = last_day_return > 0
actual_up = actual_return > 0
baseline = pd.DataFrame({
    'previous_session_date': spy.index[issue_positions - 1],
    'issue_date': calendar.issue_date.to_numpy(),
    'target_date': calendar.target_date.to_numpy(),
    'last_observed_return': last_day_return,
    'actual_return': actual_return,
    'predicted_direction': np.where(predicted_up, 'UP', 'DOWN'),
    'actual_direction': np.where(actual_up, 'UP', 'DOWN'),
    'correct': predicted_up == actual_up,
})
assert (baseline.previous_session_date < baseline.issue_date).all()
assert (baseline.issue_date < baseline.target_date).all()
display(baseline.head(10))

,previous_session_date,issue_date,target_date,last_observed_return,actual_return,predicted_direction,actual_direction,correct
0,2023-12-29,2024-01-02,2024-01-03,-0.005622,-0.008206,DOWN,DOWN,True
1,2024-01-02,2024-01-03,2024-01-04,-0.008206,-0.003222,DOWN,DOWN,True
2,2024-01-03,2024-01-04,2024-01-05,-0.003222,0.001369,DOWN,UP,False
3,2024-01-04,2024-01-05,2024-01-08,0.001369,0.014179,UP,UP,True
4,2024-01-05,2024-01-08,2024-01-09,0.014179,-0.001524,UP,DOWN,False
5,2024-01-08,2024-01-09,2024-01-10,-0.001524,0.005650,DOWN,UP,False
6,2024-01-09,2024-01-10,2024-01-11,0.005650,-0.000455,UP,DOWN,False
7,2024-01-10,2024-01-11,2024-01-12,-0.000455,0.000694,DOWN,UP,False
8,2024-01-11,2024-01-12,2024-01-16,0.000694,-0.003668,UP,DOWN,False
9,2024-01-12,2024-01-16,2024-01-17,-0.003668,-0.005582,DOWN,DOWN,True


## 3. Accuracy and direction counts

Direction accuracy is `correct predictions / all evaluated days`. The confusion table shows actual directions in rows and predicted directions in columns. The always-up and always-down rules provide simple context on the same dates.

In [3]:
n = len(baseline)
correct_days = int(baseline.correct.sum())
accuracy = correct_days / n
up_recall = float(predicted_up[actual_up].mean()) if actual_up.any() else np.nan
down_recall = float((~predicted_up[~actual_up]).mean()) if (~actual_up).any() else np.nan
metrics = pd.DataFrame([{
    'method': 'Repeat previous trading day direction',
    'n': n, 'correct_days': correct_days, 'incorrect_days': n - correct_days,
    'direction_accuracy': accuracy,
    'predicted_up_days': int(predicted_up.sum()),
    'predicted_down_days': int((~predicted_up).sum()),
    'actual_up_days': int(actual_up.sum()),
    'actual_down_days': int((~actual_up).sum()),
    'up_recall': up_recall, 'down_recall': down_recall,
    'balanced_accuracy': (up_recall + down_recall) / 2,
}])
assert metrics.loc[0, 'predicted_up_days'] + metrics.loc[0, 'predicted_down_days'] == n
assert metrics.loc[0, 'actual_up_days'] + metrics.loc[0, 'actual_down_days'] == n
print(f'Direction accuracy: {accuracy:.2%} ({correct_days} correct out of {n} days)')
print(f'Predicted UP: {int(predicted_up.sum())} days; DOWN: {int((~predicted_up).sum())} days')
print(f'Actual UP: {int(actual_up.sum())} days; DOWN (including zero returns): {int((~actual_up).sum())} days')
display(metrics)
confusion = pd.crosstab(baseline.actual_direction, baseline.predicted_direction,
                       rownames=['Actual'], colnames=['Predicted']).reindex(
                           index=['UP', 'DOWN'], columns=['UP', 'DOWN'], fill_value=0)
display(confusion)
comparison = pd.DataFrame([
    {'method': 'Repeat previous trading day', 'correct_days': correct_days, 'direction_accuracy': accuracy},
    {'method': 'Always UP', 'correct_days': int(actual_up.sum()), 'direction_accuracy': float(actual_up.mean())},
    {'method': 'Always DOWN', 'correct_days': int((~actual_up).sum()), 'direction_accuracy': float((~actual_up).mean())},
])
display(comparison)
print(f'Previous-day accuracy minus always-UP accuracy: {100 * (accuracy - actual_up.mean()):+.2f} percentage points')

Direction accuracy: 50.97% (158 correct out of 310 days)
Predicted UP: 179 days; DOWN: 131 days
Actual UP: 179 days; DOWN (including zero returns): 131 days


,method,n,correct_days,incorrect_days,direction_accuracy,predicted_up_days,predicted_down_days,actual_up_days,actual_down_days,up_recall,down_recall,balanced_accuracy
0,Repeat previous trading day direction,310,158,152,0.509677,179,131,179,131,0.575419,0.419847,0.497633


Predicted,UP,DOWN
Actual,,
UP,103,76
DOWN,76,55


,method,correct_days,direction_accuracy
0,Repeat previous trading day,158,0.509677
1,Always UP,179,0.577419
2,Always DOWN,131,0.422581


Previous-day accuracy minus always-UP accuracy: -6.77 percentage points


## 4. Save the result

The CSVs retain one prediction per target session, summary metrics, a confusion table, and the simple baseline comparison. These results describe the saved historical test calendar.

In [4]:
OUTPUT.mkdir(parents=True, exist_ok=True)
baseline.to_csv(OUTPUT / 'predictions.csv', index=False)
metrics.to_csv(OUTPUT / 'metrics.csv', index=False)
confusion.to_csv(OUTPUT / 'confusion_matrix.csv')
comparison.to_csv(OUTPUT / 'comparison.csv', index=False)
run_info = {
    'method': 'Repeat previous trading day direction',
    'spy_file': str(SPY_FILE), 'calendar_file': str(CALENDAR_FILE),
    'issue_first': str(baseline.issue_date.min().date()),
    'issue_last': str(baseline.issue_date.max().date()),
    'target_first': str(baseline.target_date.min().date()),
    'target_last': str(baseline.target_date.max().date()),
    'prediction_information': 'Close-to-close return ending at the issue close',
    'direction_rule': 'Positive return = UP; zero or negative return = DOWN',
    'days': n, 'correct_days': correct_days, 'direction_accuracy': accuracy,
}
(OUTPUT / 'run_info.json').write_text(json.dumps(run_info, indent=2), encoding='utf-8')
print('Saved results in:', OUTPUT)

Saved results in: C:\Users\Administrator\我的云端硬盘\My_Trading_System\Signature_Price_Forecast\results\previous_day_direction_baseline
